In [1]:
import os

# 1. Define the path to our good Java
java21_path = "/usr/lib/jvm/java-21-openjdk-amd64"

# 2. Set JAVA_HOME
os.environ["JAVA_HOME"] = java21_path

# 3. The Magic: Shove Java 21 to the absolute front of the PATH
os.environ["PATH"] = f"{java21_path}/bin:" + os.environ.get("PATH", "")

In [ ]:
# updated conf object, and needed to stop existing Context, to then recreate it.
# # sc.stop()

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

# 1. Force the PySpark driver to allow the legacy Security Manager
os.environ['PYSPARK_SUBMIT_ARGS'] = "--driver-java-options '-Djava.security.manager=allow' pyspark-shell"

# 1.2. Automatically find the hidden ADC file generated by gcloud
credentials_location = os.path.expanduser('~/.config/gcloud/application_default_credentials.json')

# 2. Set up your Spark configuration
conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.sql.session.timeZone", "Asia/Ho_Chi_Minh")  \
    .set("spark.sql.files.maxPartitionBytes", 16 * 1024 * 1024) \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.26.jar") \
    .set("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .set("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .set("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .set("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS") \
    .set("spark.hadoop.fs.gs.auth.type", "SERVICE_ACCOUNT_JSON_KEYFILE") \
    .set("spark.hadoop.fs.gs.auth.service.account.json.keyfile", credentials_location)

# 3. Initialize the context
sc = SparkContext(conf=conf)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/14 14:00:00 WARN Utils: Your hostname, codespaces-472bba, resolves to a loopback address: 127.0.0.1; using 10.0.13.99 instead (on interface eth0)
26/03/14 14:00:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/14 14:00:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [4]:
df_green = spark.read \
    .option("recursiveFileLookup", "True") \
    .parquet('gs://de-zc-pmg-data/pq/green')

In [17]:
df_green.rdd.getNumPartitions()

3

In [5]:
df_green.show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-23 20:10:15|  2020-01-23 20:38:16|                 N|         1|          74|         130|              1|        12.77|       36.0|  0.0|    0.

In [6]:
df_green.count()

2304517

In [16]:
len(df_green.inputFiles())

77

In [10]:
df_green.repartition(96)

DataFrame[VendorID: int, lpep_pickup_datetime: timestamp, lpep_dropoff_datetime: timestamp, store_and_fwd_flag: string, RatecodeID: int, PULocationID: int, DOLocationID: int, passenger_count: int, trip_distance: double, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, ehail_fee: double, improvement_surcharge: double, total_amount: double, payment_type: int, trip_type: int, congestion_surcharge: double]

In [11]:
df_green.count()

2304517

In [12]:
df_green.repartition(96).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(96), REPARTITION_BY_NUM, [plan_id=101]
   +- FileScan parquet [VendorID#0,lpep_pickup_datetime#1,lpep_dropoff_datetime#2,store_and_fwd_flag#3,RatecodeID#4,PULocationID#5,DOLocationID#6,passenger_count#7,trip_distance#8,fare_amount#9,extra#10,mta_tax#11,tip_amount#12,tolls_amount#13,ehail_fee#14,improvement_surcharge#15,total_amount#16,payment_type#17,trip_type#18,congestion_surcharge#19] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[gs://de-zc-pmg-data/pq/green], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,lpep_pickup_datetime:timestamp,lpep_dropoff_datetime:timestamp,store_and_fwd_...




In [ ]:
df2 = df_green.repartition(96)
df2.cache()
df2.count()

2304517

In [15]:
df2.count()

2304517